# Smart Office Occupancy Detection

An?lisis de lecturas ambientales para identificar si una oficina est? ocupada. El proyecto combina exploraci?n de datos, comparaci?n de clasificadores y ajuste de hiperpar?metros con un conjunto de prueba reservado.

**Objetivo de referencia:** evaluar si es posible alcanzar al menos 97 % de accuracy en los datos de validaci?n. Esta m?trica no garantiza el mismo rendimiento en otros edificios o ubicaciones.


## Datos y variables

Se utiliza el conjunto **Occupancy Detection** del UCI Machine Learning Repository. Las variables predictoras son temperatura, humedad, intensidad de luz, CO? y raz?n de humedad; la etiqueta indica la ocupaci?n (`1`: ocupado, `0`: vac?o).


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.occupancy_detection import (
    evaluate_baselines,
    load_dataset,
    plot_eda,
    tune_knn,
)

data = load_dataset()
print(f"Observaciones descargadas: {len(data):,}")
data.head()


## Preparaci?n y calidad de los datos

Se inspeccionan los valores faltantes y se excluyen las filas incompletas antes del an?lisis. La partici?n de entrenamiento y prueba se estratifica para conservar la proporci?n de las clases.


In [ ]:
missing_values = data.isna().sum()
print("Valores faltantes por variable:")
print(missing_values)

analysis_data = data.dropna().copy()
print(f"\nObservaciones utilizadas: {len(analysis_data):,}")


## An?lisis exploratorio

El boxplot compara la distribuci?n de CO? seg?n la ocupaci?n. El mapa de calor y el resumen de correlaciones permiten identificar qu? lecturas presentan una asociaci?n lineal m?s fuerte con la etiqueta.


In [ ]:
figure, sensor_correlations = plot_eda(
    analysis_data,
    output_dir=PROJECT_ROOT / "reports" / "figures",
)
print("Correlaci?n de cada sensor con la ocupaci?n (ordenada por magnitud):")
print(sensor_correlations)
plt.show()


## Comparaci?n de modelos

Se comparan regresi?n log?stica, ?rbol de decisi?n y K vecinos m?s cercanos (KNN) mediante validaci?n cruzada estratificada de cinco particiones en el conjunto de entrenamiento. El escalado de las variables se realiza dentro de cada pipeline, evitando que la informaci?n de una partici?n de validaci?n influya en su transformaci?n.


In [ ]:
baseline_results = evaluate_baselines(analysis_data, cv=5)
baseline_results


## Ajuste y evaluaci?n final

Se optimiza el n?mero de vecinos del modelo KNN usando `GridSearchCV` y validaci?n cruzada en el conjunto de entrenamiento. La m?trica final se calcula en el conjunto de prueba reservado, que no participa en el ajuste.


In [ ]:
knn_search, test_accuracy = tune_knn(analysis_data, cv=5)

print(f"Mejor configuraci?n: {knn_search.best_params_}")
print(f"Accuracy media en validaci?n cruzada: {knn_search.best_score_:.4f}")
print(f"Accuracy en el conjunto de prueba: {test_accuracy:.4f}")


## Conclusiones

La intensidad de luz muestra la mayor correlaci?n absoluta con la ocupaci?n en este conjunto de datos. KNN obtiene el mejor resultado promedio entre los modelos comparados; el ajuste de hiperpar?metros y la prueba independiente permiten comprobar el desempe?o sin utilizar el conjunto de prueba durante la selecci?n del modelo.

Los resultados caracterizan este dataset y su esquema de partici?n. Para valorar el uso en otras oficinas, se requiere una evaluaci?n adicional con datos independientes de edificios, plantas o periodos no vistos.

## Referencias

- Candanedo, L. M. y Feldheim, V. (2016). *Accurate occupancy detection of an office room from light, temperature, humidity and CO? measurements using statistical learning models*. Energy and Buildings, 112, 28?39. <https://doi.org/10.1016/j.enbuild.2015.11.071>
- UCI Machine Learning Repository, [Occupancy Detection](https://archive.ics.uci.edu/dataset/357/occupancy+detection).
